In [3]:
import os
import csv
import time
import random
import shutil
import numpy as np
from PIL import Image, ImageOps, ImageEnhance
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler


# =========================================================
# 0) 設定區：你主要改這裡
# =========================================================
LOCO_ROOT = r"./data\loco_1537_clean"
OUT_ROOT = r"./data\unet_loco_official_v2"

RUN_FOLDS = [
    "fold_test_C1",
    "fold_test_C2",
    "fold_test_C3",
    "fold_test_C4",
    "fold_test_C5",
    "fold_test_C6",
]

RESET_FOLDS = set()
SKIP_COMPLETED_FOLDS = True
DELETE_CHECKPOINT_AFTER_FINISH = False

IMAGE_SIZE = (352, 352)   # (W, H)
BATCH_SIZE = 4
NUM_EPOCHS = 100
LR = 3e-4
WEIGHT_DECAY = 1e-4

EARLY_STOPPING_PATIENCE = 12
SCHEDULER_PATIENCE = 4
SCHEDULER_FACTOR = 0.5
MIN_LR = 1e-6

THRESHOLD = 0.5
EPS = 1e-7

NUM_WORKERS = 0
PIN_MEMORY = True
USE_AMP = True
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUT_ROOT, exist_ok=True)


# =========================================================
# 1) 固定亂數種子
# =========================================================
def set_seed(seed=42):
    """
    固定 Python、NumPy、PyTorch 的亂數種子，讓結果比較可重現。
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =========================================================
# 2) 印出執行資訊
# =========================================================
print("========================================")
print("DEVICE =", DEVICE)
if torch.cuda.is_available():
    print("GPU =", torch.cuda.get_device_name(0))
print("OUT_ROOT =", OUT_ROOT)
print("RUN_FOLDS =", RUN_FOLDS)
print("RESET_FOLDS =", RESET_FOLDS)
print("IMAGE_SIZE =", IMAGE_SIZE)
print("BATCH_SIZE =", BATCH_SIZE)
print("NUM_EPOCHS =", NUM_EPOCHS)
print("LR =", LR)
print("WEIGHT_DECAY =", WEIGHT_DECAY)
print("EARLY_STOPPING_PATIENCE =", EARLY_STOPPING_PATIENCE)
print("USE_AMP =", USE_AMP)
print("========================================")


# =========================================================
# 3) 純 PIL 版前處理 / augmentation
# =========================================================
def resize_pair(img, mask, image_size):
    """
    將 image 和 mask 同步 resize。
    image 用雙線性插值，mask 用最近鄰插值。
    """
    img = img.resize(image_size, Image.BILINEAR)
    mask = mask.resize(image_size, Image.NEAREST)
    return img, mask


def random_hflip_pair(img, mask, p=0.5):
    """
    隨機水平翻轉 image 和 mask。
    """
    if random.random() < p:
        img = ImageOps.mirror(img)
        mask = ImageOps.mirror(mask)
    return img, mask


def random_vflip_pair(img, mask, p=0.5):
    """
    隨機垂直翻轉 image 和 mask。
    """
    if random.random() < p:
        img = ImageOps.flip(img)
        mask = ImageOps.flip(mask)
    return img, mask


def random_rotate_pair(img, mask, p=0.5, angle_limit=15):
    """
    隨機旋轉 image 和 mask，角度同步。
    """
    if random.random() < p:
        angle = random.uniform(-angle_limit, angle_limit)
        img = img.rotate(angle, resample=Image.BILINEAR, fillcolor=(0, 0, 0))
        mask = mask.rotate(angle, resample=Image.NEAREST, fillcolor=0)
    return img, mask


def random_scale_shift_pair(img, mask, p=0.5, scale_low=0.90, scale_high=1.10):
    """
    近似 ShiftScaleRotate 的純 PIL 版本：
    - scale > 1：放大後裁切
    - scale < 1：縮小後貼回黑底 canvas，形成平移效果
    """
    if random.random() >= p:
        return img, mask

    w, h = img.size
    scale = random.uniform(scale_low, scale_high)

    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))

    img_scaled = img.resize((new_w, new_h), Image.BILINEAR)
    mask_scaled = mask.resize((new_w, new_h), Image.NEAREST)

    if scale >= 1.0:
        max_left = max(0, new_w - w)
        max_top = max(0, new_h - h)

        left = random.randint(0, max_left) if max_left > 0 else 0
        top = random.randint(0, max_top) if max_top > 0 else 0

        img_out = img_scaled.crop((left, top, left + w, top + h))
        mask_out = mask_scaled.crop((left, top, left + w, top + h))
        return img_out, mask_out

    canvas_img = Image.new("RGB", (w, h), (0, 0, 0))
    canvas_mask = Image.new("L", (w, h), 0)

    max_left = max(0, w - new_w)
    max_top = max(0, h - new_h)

    left = random.randint(0, max_left) if max_left > 0 else 0
    top = random.randint(0, max_top) if max_top > 0 else 0

    canvas_img.paste(img_scaled, (left, top))
    canvas_mask.paste(mask_scaled, (left, top))

    return canvas_img, canvas_mask


def random_brightness_contrast(img, p=0.4, brightness_limit=0.15, contrast_limit=0.15):
    """
    隨機調整 image 的亮度與對比，不改 mask。
    """
    if random.random() < p:
        brightness_factor = random.uniform(1.0 - brightness_limit, 1.0 + brightness_limit)
        img = ImageEnhance.Brightness(img).enhance(brightness_factor)

    if random.random() < p:
        contrast_factor = random.uniform(1.0 - contrast_limit, 1.0 + contrast_limit)
        img = ImageEnhance.Contrast(img).enhance(contrast_factor)

    return img


def preprocess_pair(img, mask, image_size=(352, 352), is_train=False):
    """
    將 image / mask 做同步前處理。

    步驟：
    1. resize
    2. train 時做 augmentation
    3. image normalize 到 [0,1]
    4. mask 二值化成 0/1
    5. image HWC -> CHW
    6. mask HW -> 1HW
    """
    img, mask = resize_pair(img, mask, image_size)

    if is_train:
        img, mask = random_hflip_pair(img, mask, p=0.5)
        img, mask = random_vflip_pair(img, mask, p=0.5)
        img, mask = random_rotate_pair(img, mask, p=0.5, angle_limit=15)
        img, mask = random_scale_shift_pair(img, mask, p=0.5, scale_low=0.90, scale_high=1.10)
        img = random_brightness_contrast(img, p=0.4, brightness_limit=0.15, contrast_limit=0.15)

    img_np = np.array(img, dtype=np.float32) / 255.0
    mask_np = np.array(mask, dtype=np.uint8)

    mask_np = (mask_np > 127).astype(np.float32)

    img_np = np.transpose(img_np, (2, 0, 1))
    mask_np = np.expand_dims(mask_np, axis=0)

    return img_np, mask_np


# =========================================================
# 4) Dataset
# =========================================================
class PolypDataset(Dataset):
    """
    從 csv 讀取資料，回傳 image / mask tensor 與對應資訊。
    """
    def __init__(self, csv_file, image_size=(352, 352), is_train=False):
        self.rows = []
        self.image_size = image_size
        self.is_train = is_train

        with open(csv_file, "r", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.rows.append(row)

    def __len__(self):
        """
        回傳資料集筆數。
        """
        return len(self.rows)

    def __getitem__(self, idx):
        """
        讀取單筆資料並完成前處理。
        """
        row = self.rows[idx]

        img = Image.open(row["image_path"]).convert("RGB")
        mask = Image.open(row["mask_path"]).convert("L")

        img_np, mask_np = preprocess_pair(
            img=img,
            mask=mask,
            image_size=self.image_size,
            is_train=self.is_train
        )

        return {
            "image": torch.tensor(img_np, dtype=torch.float32),
            "mask": torch.tensor(mask_np, dtype=torch.float32),
            "stem": row["stem"],
            "image_path": row["image_path"],
            "mask_path": row["mask_path"],
            "center": row["center"] if "center" in row else "UNKNOWN",
        }


# =========================================================
# 5) U-Net 模型
# =========================================================
class DoubleConv(nn.Module):
    """
    U-Net 常用雙卷積模組。
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        """
        將輸入特徵送入兩層卷積。
        """
        return self.block(x)


class UNet(nn.Module):
    """
    標準 U-Net，base channel = 32。
    """
    def __init__(self, in_channels=3, out_channels=1, base_ch=32):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, base_ch)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(base_ch, base_ch * 2)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(base_ch * 2, base_ch * 4)
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = DoubleConv(base_ch * 4, base_ch * 8)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(base_ch * 8, base_ch * 16)

        self.up4 = nn.ConvTranspose2d(base_ch * 16, base_ch * 8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(base_ch * 16, base_ch * 8)

        self.up3 = nn.ConvTranspose2d(base_ch * 8, base_ch * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(base_ch * 8, base_ch * 4)

        self.up2 = nn.ConvTranspose2d(base_ch * 4, base_ch * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base_ch * 4, base_ch * 2)

        self.up1 = nn.ConvTranspose2d(base_ch * 2, base_ch, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base_ch * 2, base_ch)

        self.out_conv = nn.Conv2d(base_ch, out_channels, kernel_size=1)

    def forward(self, x):
        """
        U-Net 前向傳播。
        """
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        b = self.bottleneck(self.pool4(e4))

        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out_conv(d1)


# =========================================================
# 6) Loss：BCE + Dice
# =========================================================
class DiceBCELoss(nn.Module):
    """
    BCEWithLogitsLoss + Dice Loss。
    """
    def __init__(self, smooth=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.smooth = smooth

    def forward(self, logits, targets):
        """
        計算 BCE 與 Dice Loss 的總和。
        """
        bce = self.bce(logits, targets)

        probs = torch.sigmoid(logits)
        probs = probs.view(probs.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        intersection = (probs * targets).sum(dim=1)
        dice = (2.0 * intersection + self.smooth) / (
            probs.sum(dim=1) + targets.sum(dim=1) + self.smooth
        )
        dice_loss = 1.0 - dice.mean()

        return bce + dice_loss


# =========================================================
# 7) Metrics
# =========================================================
def batch_metrics_from_logits(logits, targets, threshold=0.5, eps=1e-7):
    """
    從 logits 與 GT 計算 batch 平均：
    - Dice
    - IoU
    - Precision
    - Recall
    """
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    tp = (preds * targets).sum(dim=1)
    fp = (preds * (1 - targets)).sum(dim=1)
    fn = ((1 - preds) * targets).sum(dim=1)

    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)

    return {
        "dice": dice.mean().item(),
        "iou": iou.mean().item(),
        "precision": precision.mean().item(),
        "recall": recall.mean().item(),
    }


# =========================================================
# 8) train / valid / test 單 epoch
# =========================================================
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None):
    """
    執行單個 epoch。
    有 optimizer 時表示 train，否則表示 valid/test。
    這個函式不顯示 batch 進度條，只回傳該 epoch 的整體指標。
    """
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    total_precision = 0.0
    total_recall = 0.0
    total_count = 0

    with torch.set_grad_enabled(is_train):
        for batch in loader:
            images = batch["image"].to(DEVICE, non_blocking=True)
            masks = batch["mask"].to(DEVICE, non_blocking=True)

            if is_train:
                optimizer.zero_grad()

            with autocast(enabled=USE_AMP):
                logits = model(images)
                loss = criterion(logits, masks)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            batch_size = images.size(0)
            metrics = batch_metrics_from_logits(logits, masks, THRESHOLD, EPS)

            total_loss += loss.item() * batch_size
            total_dice += metrics["dice"] * batch_size
            total_iou += metrics["iou"] * batch_size
            total_precision += metrics["precision"] * batch_size
            total_recall += metrics["recall"] * batch_size
            total_count += batch_size

    return {
        "loss": total_loss / total_count,
        "dice": total_dice / total_count,
        "iou": total_iou / total_count,
        "precision": total_precision / total_count,
        "recall": total_recall / total_count,
    }


# =========================================================
# 9) 存 test prediction 影像
# =========================================================
def save_test_predictions(model, loader, save_dir, threshold=0.5):
    """
    儲存 test set 的原圖、GT、預測結果。
    """
    model.eval()

    image_dir = os.path.join(save_dir, "image")
    gt_dir = os.path.join(save_dir, "mask_gt")
    pred_dir = os.path.join(save_dir, "mask_pred")

    os.makedirs(image_dir, exist_ok=True)
    os.makedirs(gt_dir, exist_ok=True)
    os.makedirs(pred_dir, exist_ok=True)

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            masks = batch["mask"].to(DEVICE)
            stems = batch["stem"]
            image_paths = batch["image_path"]

            with autocast(enabled=USE_AMP):
                logits = model(images)

            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()

            masks_np = masks.cpu().numpy()
            preds_np = preds.cpu().numpy()

            for i in range(images.size(0)):
                stem = stems[i]

                raw_img = Image.open(image_paths[i]).convert("RGB")
                raw_img = raw_img.resize(IMAGE_SIZE, Image.BILINEAR)
                raw_img.save(os.path.join(image_dir, f"{stem}.png"))

                gt_mask = (masks_np[i, 0] * 255).astype(np.uint8)
                Image.fromarray(gt_mask).save(os.path.join(gt_dir, f"{stem}.png"))

                pred_mask = (preds_np[i, 0] * 255).astype(np.uint8)
                Image.fromarray(pred_mask).save(os.path.join(pred_dir, f"{stem}.png"))


# =========================================================
# 10) checkpoint 工具
# =========================================================
def save_checkpoint(
    checkpoint_path,
    epoch,
    model,
    optimizer,
    scheduler,
    best_val_dice,
    best_val_iou,
    best_epoch,
    history,
    epochs_no_improve
):
    """
    儲存 checkpoint，方便中斷續跑。
    """
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_dice": best_val_dice,
        "best_val_iou": best_val_iou,
        "best_epoch": best_epoch,
        "history": history,
        "epochs_no_improve": epochs_no_improve,
    }, checkpoint_path)


def load_checkpoint(checkpoint_path, model, optimizer, scheduler):
    """
    載入 checkpoint，回傳續跑資訊。
    """
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    return (
        ckpt["epoch"] + 1,
        ckpt["best_val_dice"],
        ckpt["best_val_iou"],
        ckpt["best_epoch"],
        ckpt["history"],
        ckpt["epochs_no_improve"]
    )


# =========================================================
# 11) 主程式：逐 fold 訓練
# =========================================================
set_seed(SEED)

all_results = []

for fold_name in RUN_FOLDS:
    print("========================================")
    print("Training", fold_name)
    print("========================================")

    fold_dir = os.path.join(LOCO_ROOT, fold_name)
    save_dir = os.path.join(OUT_ROOT, fold_name)
    pred_save_dir = os.path.join(save_dir, "test_predictions")
    result_txt = os.path.join(save_dir, "result.txt")
    checkpoint_path = os.path.join(save_dir, "checkpoint_latest.pth")
    best_model_path = os.path.join(save_dir, "best_model.pth")

    if SKIP_COMPLETED_FOLDS and os.path.exists(result_txt) and fold_name not in RESET_FOLDS:
        print(f"[{fold_name}] already completed -> skip")
        print()
        continue

    if fold_name in RESET_FOLDS and os.path.exists(save_dir):
        print(f"[{fold_name}] RESET -> remove old folder")
        shutil.rmtree(save_dir)

    os.makedirs(save_dir, exist_ok=True)

    train_csv = os.path.join(fold_dir, "train.csv")
    val_csv = os.path.join(fold_dir, "val.csv")
    test_csv = os.path.join(fold_dir, "test.csv")

    train_dataset = PolypDataset(train_csv, IMAGE_SIZE, is_train=True)
    val_dataset = PolypDataset(val_csv, IMAGE_SIZE, is_train=False)
    test_dataset = PolypDataset(test_csv, IMAGE_SIZE, is_train=False)

    print(f"train={len(train_dataset)} | val={len(val_dataset)} | test={len(test_dataset)}")

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY if DEVICE == "cuda" else False
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY if DEVICE == "cuda" else False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY if DEVICE == "cuda" else False
    )

    model = UNet(in_channels=3, out_channels=1, base_ch=32).to(DEVICE)
    criterion = DiceBCELoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=MIN_LR
    )

    scaler = GradScaler(enabled=USE_AMP)

    best_val_dice = -1.0
    best_val_iou = -1.0
    best_epoch = -1
    history = []
    start_epoch = 1
    epochs_no_improve = 0

    if os.path.exists(checkpoint_path) and fold_name not in RESET_FOLDS:
        start_epoch, best_val_dice, best_val_iou, best_epoch, history, epochs_no_improve = load_checkpoint(
            checkpoint_path, model, optimizer, scheduler
        )
        print(f"[{fold_name}] resume from epoch {start_epoch}")
    else:
        print(f"[{fold_name}] start from scratch")

    start_time = time.time()

    # 你要的：只有 epoch 進度條
    epoch_pbar = tqdm(
        range(start_epoch, NUM_EPOCHS + 1),
        desc=fold_name,
        leave=True,
        ncols=160
    )

    for epoch in epoch_pbar:
        train_metrics = run_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler
        )

        val_metrics = run_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            optimizer=None,
            scaler=scaler
        )

        current_lr = optimizer.param_groups[0]["lr"]

        history.append([
            epoch,
            current_lr,
            train_metrics["loss"], train_metrics["dice"], train_metrics["iou"],
            train_metrics["precision"], train_metrics["recall"],
            val_metrics["loss"], val_metrics["dice"], val_metrics["iou"],
            val_metrics["precision"], val_metrics["recall"]
        ])

        scheduler.step(val_metrics["dice"])

        if val_metrics["dice"] > best_val_dice:
            best_val_dice = val_metrics["dice"]
            best_val_iou = val_metrics["iou"]
            best_epoch = epoch
            epochs_no_improve = 0

            torch.save(model.state_dict(), best_model_path)
        else:
            epochs_no_improve += 1

        save_checkpoint(
            checkpoint_path=checkpoint_path,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            best_val_dice=best_val_dice,
            best_val_iou=best_val_iou,
            best_epoch=best_epoch,
            history=history,
            epochs_no_improve=epochs_no_improve
        )

        # 你要看的 epoch 指標全部放在這裡
        epoch_pbar.set_postfix({
            "lr": f"{current_lr:.2e}",
            "tr_dice": f"{train_metrics['dice']:.4f}",
            "va_dice": f"{val_metrics['dice']:.4f}",
            "tr_iou": f"{train_metrics['iou']:.4f}",
            "va_iou": f"{val_metrics['iou']:.4f}",
            "best": f"{best_val_dice:.4f}",
            "no_imp": f"{epochs_no_improve}/{EARLY_STOPPING_PATIENCE}",
        })

        # 每個 epoch 再印一行完整資訊
        print(
            f"[{fold_name}] Epoch {epoch:03d}/{NUM_EPOCHS} | "
            f"lr={current_lr:.6e} | "
            f"train_loss={train_metrics['loss']:.4f} "
            f"train_dice={train_metrics['dice']:.4f} "
            f"train_iou={train_metrics['iou']:.4f} "
            f"train_precision={train_metrics['precision']:.4f} "
            f"train_recall={train_metrics['recall']:.4f} | "
            f"valid_loss={val_metrics['loss']:.4f} "
            f"valid_dice={val_metrics['dice']:.4f} "
            f"valid_iou={val_metrics['iou']:.4f} "
            f"valid_precision={val_metrics['precision']:.4f} "
            f"valid_recall={val_metrics['recall']:.4f} | "
            f"best_valid_dice={best_val_dice:.4f} @ epoch {best_epoch}"
        )

        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"[{fold_name}] Early stopping triggered.")
            break

    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

    test_metrics = run_one_epoch(
        model=model,
        loader=test_loader,
        criterion=criterion,
        optimizer=None,
        scaler=scaler
    )

    save_test_predictions(model, test_loader, pred_save_dir, THRESHOLD)

    elapsed = time.time() - start_time

    delta_shift_dice = best_val_dice - test_metrics["dice"]
    delta_shift_iou = best_val_iou - test_metrics["iou"]

    with open(os.path.join(save_dir, "history.csv"), "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch", "lr",
            "train_loss", "train_dice", "train_iou", "train_precision", "train_recall",
            "valid_loss", "valid_dice", "valid_iou", "valid_precision", "valid_recall"
        ])
        writer.writerows(history)

    with open(result_txt, "w", encoding="utf-8") as f:
        f.write(f"fold: {fold_name}\n")
        f.write(f"best_epoch: {best_epoch}\n")
        f.write(f"best_val_dice: {best_val_dice:.6f}\n")
        f.write(f"best_val_iou: {best_val_iou:.6f}\n")
        f.write(f"test_loss: {test_metrics['loss']:.6f}\n")
        f.write(f"test_dice: {test_metrics['dice']:.6f}\n")
        f.write(f"test_iou: {test_metrics['iou']:.6f}\n")
        f.write(f"test_precision: {test_metrics['precision']:.6f}\n")
        f.write(f"test_recall: {test_metrics['recall']:.6f}\n")
        f.write(f"delta_shift_dice: {delta_shift_dice:.6f}\n")
        f.write(f"delta_shift_iou: {delta_shift_iou:.6f}\n")
        f.write(f"time_sec: {elapsed:.2f}\n")
        f.write(f"prediction_dir: {pred_save_dir}\n")

    if DELETE_CHECKPOINT_AFTER_FINISH and os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    all_results.append([
        fold_name,
        best_epoch,
        best_val_dice,
        best_val_iou,
        test_metrics["loss"],
        test_metrics["dice"],
        test_metrics["iou"],
        test_metrics["precision"],
        test_metrics["recall"],
        delta_shift_dice,
        delta_shift_iou,
        elapsed
    ])

    # fold 結束後，test 指標完整顯示
    print("------------------------------------------------------------")
    print(f"[{fold_name}] TEST RESULT")
    print(f"best_epoch       = {best_epoch}")
    print(f"best_val_dice    = {best_val_dice:.4f}")
    print(f"best_val_iou     = {best_val_iou:.4f}")
    print(f"test_loss        = {test_metrics['loss']:.4f}")
    print(f"test_dice        = {test_metrics['dice']:.4f}")
    print(f"test_iou         = {test_metrics['iou']:.4f}")
    print(f"test_precision   = {test_metrics['precision']:.4f}")
    print(f"test_recall      = {test_metrics['recall']:.4f}")
    print(f"delta_shift_dice = {delta_shift_dice:.4f}")
    print(f"delta_shift_iou  = {delta_shift_iou:.4f}")
    print(f"prediction_dir   = {pred_save_dir}")
    print("------------------------------------------------------------")
    print()


# =========================================================
# 12) 總結果輸出
# =========================================================
summary_csv = os.path.join(OUT_ROOT, "all_fold_results.csv")
with open(summary_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow([
        "fold", "best_epoch", "best_val_dice", "best_val_iou",
        "test_loss", "test_dice", "test_iou", "test_precision", "test_recall",
        "delta_shift_dice", "delta_shift_iou", "time_sec"
    ])
    writer.writerows(all_results)

if len(all_results) > 0:
    test_dices = [row[5] for row in all_results]
    test_ious = [row[6] for row in all_results]

    mean_test_dice = sum(test_dices) / len(test_dices)
    mean_test_iou = sum(test_ious) / len(test_ious)

    worst_row = min(all_results, key=lambda x: x[5])
    best_row = max(all_results, key=lambda x: x[5])

    worst_center = worst_row[0]
    worst_test_dice = worst_row[5]

    best_center = best_row[0]
    best_test_dice = best_row[5]

    center_gap_dice = best_test_dice - worst_test_dice

    summary_txt = os.path.join(OUT_ROOT, "summary.txt")
    with open(summary_txt, "w", encoding="utf-8") as f:
        f.write("========================================\n")
        f.write("LOCO Summary\n")
        f.write("========================================\n")
        for row in all_results:
            f.write(
                f"{row[0]} | best_epoch={row[1]} | best_val_dice={row[2]:.4f} | best_val_iou={row[3]:.4f} | "
                f"test_loss={row[4]:.4f} | test_dice={row[5]:.4f} | test_iou={row[6]:.4f} | "
                f"test_precision={row[7]:.4f} | test_recall={row[8]:.4f} | "
                f"delta_shift_dice={row[9]:.4f} | delta_shift_iou={row[10]:.4f} | time_sec={row[11]:.2f}\n"
            )
        f.write("\n")
        f.write(f"mean_test_dice: {mean_test_dice:.4f}\n")
        f.write(f"mean_test_iou: {mean_test_iou:.4f}\n")
        f.write(f"best_center: {best_center}\n")
        f.write(f"best_test_dice: {best_test_dice:.4f}\n")
        f.write(f"worst_center: {worst_center}\n")
        f.write(f"worst_test_dice: {worst_test_dice:.4f}\n")
        f.write(f"center_gap_dice: {center_gap_dice:.4f}\n")
        f.write("========================================\n")

    print("========================================")
    print("Current run finished")
    print("========================================")
    print(f"summary csv: {summary_csv}")
    print(f"summary txt: {summary_txt}")

    for row in all_results:
        print(
            f"{row[0]} | best_epoch={row[1]} | best_val_dice={row[2]:.4f} | "
            f"test_dice={row[5]:.4f} | test_iou={row[6]:.4f} | "
            f"test_precision={row[7]:.4f} | test_recall={row[8]:.4f} | "
            f"delta_shift_dice={row[9]:.4f}"
        )

    print("----------------------------------------")
    print(f"mean_test_dice = {mean_test_dice:.4f}")
    print(f"mean_test_iou  = {mean_test_iou:.4f}")
    print(f"best_center    = {best_center}")
    print(f"best_test_dice = {best_test_dice:.4f}")
    print(f"worst_center   = {worst_center}")
    print(f"worst_test_dice= {worst_test_dice:.4f}")
    print(f"center_gap_dice= {center_gap_dice:.4f}")
    print("========================================")
else:
    print("沒有新的 fold 在這次執行中完成。")

DEVICE = cuda
GPU = NVIDIA GeForce RTX 3050 Laptop GPU
OUT_ROOT = ./data\unet_loco_official_v2
RUN_FOLDS = ['fold_test_C1', 'fold_test_C2', 'fold_test_C3', 'fold_test_C4', 'fold_test_C5', 'fold_test_C6']
RESET_FOLDS = set()
IMAGE_SIZE = (352, 352)
BATCH_SIZE = 4
NUM_EPOCHS = 100
LR = 0.0003
WEIGHT_DECAY = 0.0001
EARLY_STOPPING_PATIENCE = 12
USE_AMP = True
Training fold_test_C1
train=1089 | val=192 | test=256
[fold_test_C1] start from scratch


fold_test_C1:   1%|      | 1/100 [01:33<2:33:46, 93.20s/it, lr=3.00e-04, tr_dice=0.1264, va_dice=0.0489, tr_iou=0.0851, va_iou=0.0409, best=0.0489, no_imp=0/12]

[fold_test_C1] Epoch 001/100 | lr=3.000000e-04 | train_loss=1.2810 train_dice=0.1264 train_iou=0.0851 train_precision=0.3081 train_recall=0.2610 | valid_loss=1.1903 valid_dice=0.0489 valid_iou=0.0409 valid_precision=0.6834 valid_recall=0.0829 | best_valid_dice=0.0489 @ epoch 1


fold_test_C1:   2%|      | 2/100 [03:07<2:33:13, 93.81s/it, lr=3.00e-04, tr_dice=0.0810, va_dice=0.1028, tr_iou=0.0617, va_iou=0.0773, best=0.1028, no_imp=0/12]

[fold_test_C1] Epoch 002/100 | lr=3.000000e-04 | train_loss=1.1593 train_dice=0.0810 train_iou=0.0617 train_precision=0.4739 train_recall=0.1435 | valid_loss=1.1306 valid_dice=0.1028 valid_iou=0.0773 valid_precision=0.5878 valid_recall=0.1187 | best_valid_dice=0.1028 @ epoch 2


...

[fold_test_C6] Epoch 088/100 | lr=9.375000e-06 | train_loss=0.3672 train_dice=0.7832 train_iou=0.7049 train_precision=0.8170 train_recall=0.8503 | valid_loss=0.4833 valid_dice=0.6914 valid_iou=0.6075 valid_precision=0.8229 valid_recall=0.7358 | best_valid_dice=0.6958 @ epoch 78


fold_test_C6:  89%|██▋| 89/100 [2:39:31<19:04, 104.07s/it, lr=4.69e-06, tr_dice=0.7832, va_dice=0.6853, tr_iou=0.7048, va_iou=0.6017, best=0.6958, no_imp=11/12]

[fold_test_C6] Epoch 089/100 | lr=4.687500e-06 | train_loss=0.3687 train_dice=0.7832 train_iou=0.7048 train_precision=0.8206 train_recall=0.8468 | valid_loss=0.4915 valid_dice=0.6853 valid_iou=0.6017 valid_precision=0.8248 valid_recall=0.7307 | best_valid_dice=0.6958 @ epoch 78


fold_test_C6:  89%|██▋| 89/100 [2:41:15<19:55, 108.71s/it, lr=4.69e-06, tr_dice=0.7803, va_dice=0.6940, tr_iou=0.7016, va_iou=0.6094, best=0.6958, no_imp=12/12]


[fold_test_C6] Epoch 090/100 | lr=4.687500e-06 | train_loss=0.3806 train_dice=0.7803 train_iou=0.7016 train_precision=0.8225 train_recall=0.8443 | valid_loss=0.4787 valid_dice=0.6940 valid_iou=0.6094 valid_precision=0.8106 valid_recall=0.7503 | best_valid_dice=0.6958 @ epoch 78
[fold_test_C6] Early stopping triggered.
------------------------------------------------------------
[fold_test_C6] TEST RESULT
best_epoch       = 78
best_val_dice    = 0.6958
best_val_iou     = 0.6091
test_loss        = 0.5157
test_dice        = 0.7221
test_iou         = 0.6400
test_precision   = 0.8299
test_recall      = 0.7456
delta_shift_dice = -0.0264
delta_shift_iou  = -0.0309
prediction_dir   = ./data\unet_loco_official_v2\fold_test_C6\test_predictions
------------------------------------------------------------

Current run finished
summary csv: ./data\unet_loco_official_v2\all_fold_results.csv
summary txt: ./data\unet_loco_official_v2\summary.txt
fold_test_C1 | best_epoch=85 | best_val_dice=0.6914 | te